### Build a complete GPT architecture by assembling all previously built components.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Hyperparameters
VOCAB_SIZE = 5000

EMBED_DIM = 256

NUM_HEADS = 8

NUM_LAYERS = 6

BLOCK_SIZE = 128

DROPOUT = 0.1

In [2]:
import importlib

# 1. Use importlib to dynamically load the notebook 4 module
notebook04_module = importlib.import_module("ipynb.fs.full.04_TransformerBlock(GPT Decoder Block)")

# 2. Extract the class from the loaded module
TransformerBlock = notebook04_module.TransformerBlock


torch.Size([2, 10, 256])
788,992


In [ ]:
# Verify - Transformer block
block = TransformerBlock(
    EMBED_DIM,
    NUM_HEADS
)

x = torch.randn(
    2,
    128,
    EMBED_DIM
)

print(block(x).shape)

torch.Size([2, 128, 256])


## Build Mini GPT

In [ ]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        block_size,
        num_heads,
        num_layers
    ):

        super().__init__()

        self.block_size = block_size
        # Token Embedding
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            embed_dim
        )
        # Position Embedding
        self.position_embedding_table = nn.Embedding(
            block_size,
            embed_dim
        )
        # Tranformer Block
        self.blocks = nn.Sequential(

            *[
                TransformerBlock(
                    embed_dim,
                    num_heads
                )

                for _ in range(
                    num_layers
                )
            ]

        )
        # LayerNorm
        self.ln_f = nn.LayerNorm(
            embed_dim
        )
        # Language Modeling Head
        self.lm_head = nn.Linear(
            embed_dim,
            vocab_size
        )

    def forward(
        self,
        idx,
        targets=None
    ):

        B,T = idx.shape

        token_embeddings = (
            self.token_embedding_table(
                idx
            )
        )

        position_embeddings = (
            self.position_embedding_table(
                torch.arange(
                    T,
                    device=idx.device
                )
            )
        )

        x = (
            token_embeddings
            +
            position_embeddings
        )

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B,T,C = logits.shape

            logits = logits.view(
                B*T,
                C
            )

            targets = targets.view(
                B*T
            )

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

In [10]:
model = MiniGPT(

    vocab_size=VOCAB_SIZE,

    embed_dim=EMBED_DIM,

    block_size=BLOCK_SIZE,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS

)

In [11]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"{total_params:,}"
)

7,332,232


In [ ]:
# dummy test
x = torch.randint(
    0,
    VOCAB_SIZE,
    (4,128)
)

logits, loss = model(
    x,
    x
)

print(logits.shape)

print(loss)

torch.Size([512, 5000])
tensor(8.6391, grad_fn=<NllLossBackward0>)
